<a href="https://colab.research.google.com/github/dimitarpg13/agentic_architectures_and_design_patterns/blob/main/notebooks/causal_inference/causal_inference_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Causal Inference Agentic Workflow

A LangGraph-based multi-agent system that answers causal questions using
Pearl's **causal ladder** (Association → Intervention → Counterfactual).

### Architecture

```
User Question
     │
     ▼
┌─────────────┐
│ Orchestrator │  ← classifies rung, extracts CausalQuery
└─────┬───────┘
      │  L1 / L2 / L3
      ▼
┌──────────────────────────┐
│ L1 Association Agent     │  P(Y|X)
│ L2 Intervention Agent    │  P(Y|do(X))
│ L3 Counterfactual Agent  │  P(Y_x | X=x', Y=y')
└──────────┬───────────────┘
           ▼
┌───────────────┐
│   Validator   │  ← identifiability, positivity, estimator check
└───────┬───────┘
   pass │ re_route → Orchestrator (loop)
   fail │
        ▼
┌───────────────┐
│  Synthesizer  │  ← final causal report
└───────────────┘
```

See `causal_inference_agentic_workflow.md` for the full design document.

In [1]:
# Optional — uncomment to install dependencies
%pip install langgraph openai anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.0/457.0 kB 9.9 MB/s eta 0:00:00


In [3]:
from __future__ import annotations

import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from causal_inference.settings import Settings
from causal_inference.llm_client import build_llm_client
from causal_inference.graph import run_causal_workflow

print("Notebook directory:", notebook_dir)

Notebook directory: /content


## Configuration

API keys can come from the repo-root `.env` file **or** be set directly here.

```
# .env
LLM_PROVIDER=openai          # or "anthropic"
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_MODEL=gpt-4o-mini
ANTHROPIC_MODEL=claude-3-5-sonnet-latest
```

In [5]:
# --- Override here if you prefer not to use .env ---
LLM_PROVIDER = "openai"            # "openai" or "anthropic"; None → use .env
OPENAI_API_KEY_INLINE = "<your_OPENAI_API_KEY_here>"
ANTHROPIC_API_KEY_INLINE = "<your_ANTHROPIC_API_KEY_here"

settings = Settings.from_env()
if LLM_PROVIDER:
    settings = Settings(
        llm_provider=LLM_PROVIDER,
        openai_api_key=OPENAI_API_KEY_INLINE or settings.openai_api_key,
        anthropic_api_key=ANTHROPIC_API_KEY_INLINE or settings.anthropic_api_key,
        openai_model=settings.openai_model,
        anthropic_model=settings.anthropic_model,
    )

llm = build_llm_client(settings)
print(f"LLM provider: {settings.llm_provider}")

LLM provider: openai


## DAG & SCM Definitions

The agents reason over a shared DAG (and optionally an SCM for L3). Define one here
or use the defaults.

In [6]:
# -- DAG for L1 / L2 examples --
smoking_dag = {
    "nodes": ["Smoking", "Tar", "Cancer", "Genetics"],
    "edges": [
        ["Smoking", "Tar"],
        ["Tar", "Cancer"],
        ["Genetics", "Smoking"],
        ["Genetics", "Cancer"],
    ],
    "description": (
        "Smoking → Tar → Cancer (front-door path). "
        "Genetics is an unmeasured common cause of Smoking and Cancer."
    ),
}

# -- SCM for L3 example --
drug_scm = {
    "structural_equations": {
        "X": "U_X",
        "Y": "0.6 * X + U_Y",
    },
    "exogenous": {
        "U_X": "Bernoulli(0.5)",
        "U_Y": "Normal(0, 0.1)",
    },
    "description": (
        "X = took drug (binary), Y = recovery score. "
        "Structural equation: Y = 0.6*X + U_Y."
    ),
}

drug_dag = {
    "nodes": ["X", "Y"],
    "edges": [["X", "Y"]],
    "description": "Simple RCT-like DAG: X → Y with no confounders.",
}

---

## Example 1 — L1 Association

> "Is smoking correlated with cancer after controlling for genetics?"

This is an **observational / associational** question → routed to the L1 agent.

In [7]:
from IPython.display import Markdown

result_l1 = await run_causal_workflow(
    question="Is smoking correlated with cancer after controlling for genetics?",
    llm=llm,
    dag=smoking_dag,
)

print("Ladder rung:", result_l1["ladder_rung"])
Markdown(result_l1["final_report"])

Ladder rung: L1


## Causal Report

### Question
Is smoking correlated with cancer after controlling for genetics?

### Ladder Rung
L1 was addressed because the analysis focused on the association between smoking and cancer while controlling for genetics, which is a common approach in observational studies to assess potential confounding.

### Causal Query
- **Treatment**: Smoking
- **Outcome**: Cancer
- **Estimand**: P(cancer | smoking)
- **Covariates**: Genetics
- **Assumptions**: Controlling for genetics addresses confounding.

### DAG / SCM Summary
The assumed causal structure is represented by a Directed Acyclic Graph (DAG) with nodes for Smoking, Tar, Cancer, and Genetics. The edges indicate that smoking influences tar, which in turn affects cancer. Genetics is an unmeasured common cause of both smoking and cancer, suggesting that controlling for genetics may not fully account for confounding.

### Analysis
- **Method used**: Regression / partial correlation
- **Result / estimate**: The analysis would assess the correlation between smoking and cancer while controlling for genetics, potentially revealing a positive association.
- **Confidence / uncertainty**: N/A

### Assumptions
1. Controlling for genetics addresses confounding.
2. The relationship between smoking and cancer can be assessed through regression/partial correlation.

### Validation
The validation checks revealed several issues:
- **Identifiability**: The analysis fails to meet the identifiability condition due to unmeasured confounding from genetics.
- **Positivity**: The positivity condition is satisfied.
- **Estimator matches estimand**: The chosen statistical method does not estimate the stated causal estimand (P(cancer | smoking)).
- **Unmeasured confounders threat**: The presence of unmeasured confounders threatens the validity of causal claims.

### Caveats & Limitations
This analysis cannot claim causation between smoking and cancer due to the presence of unmeasured confounding factors (genetics). The results are purely associational and may be influenced by other unmeasured variables. Additionally, the chosen statistical method does not adequately address the causal estimand due to these confounding factors.

### Conclusion
While there may be an observed correlation between smoking and cancer when controlling for genetics, this analysis cannot establish a causal relationship due to the presence of unmeasured confounding. Further research with better control for confounding variables is necessary to draw more definitive conclusions.

---

## Example 2 — L2 Intervention

> "What is the causal effect of smoking on cancer?"

This is an **interventional** question — P(Cancer | do(Smoking)) → routed to the L2 agent.
The DAG has an unmeasured confounder (Genetics) but a front-door path via Tar.

In [8]:
result_l2 = await run_causal_workflow(
    question="What is the causal effect of smoking on cancer?",
    llm=llm,
    dag=smoking_dag,
)

print("Ladder rung:", result_l2["ladder_rung"])
Markdown(result_l2["final_report"])

Ladder rung: L1


## Causal Report

### Question
What is the causal effect of smoking on cancer?

### Ladder Rung
L1 was addressed because the analysis focused on establishing an association between smoking and cancer without controlling for confounding factors, which is typical for this rung.

### Causal Query
- **Treatment**: Smoking
- **Outcome**: Cancer
- **Estimand**: P(cancer | smoking)

### DAG / SCM Summary
The assumed causal structure is represented by a Directed Acyclic Graph (DAG) with the following nodes: Smoking, Tar, Cancer, and Genetics. The edges indicate that smoking leads to tar, which in turn affects cancer. Additionally, genetics is an unmeasured common cause of both smoking and cancer, suggesting that the relationship between smoking and cancer may be confounded by genetic factors.

### Analysis
- **Method used**: Regression / partial correlation
- **Result / estimate**: The analysis indicates an association between smoking and cancer; however, this association may be confounded by genetics.
- **Confidence / uncertainty**: N/A, as the analysis does not provide a confidence interval due to the unmeasured confounding.

### Assumptions
1. The analysis assumes that the relationship between smoking and cancer can be estimated without accounting for unmeasured confounders.
2. It assumes that the regression method used is appropriate for estimating the causal effect, despite the presence of confounding.

### Validation
The validation checks revealed several issues:
- The required identifiability conditions are not met because the analysis does not account for the unmeasured confounder (Genetics) that affects both smoking and cancer.
- The chosen statistical method (Regression / partial correlation) does not estimate the stated causal estimand (P(cancer | smoking)) due to the presence of unmeasured confounding.
- Positivity conditions were satisfied, but the overall analysis was deemed insufficient.

### Caveats & Limitations
This analysis cannot claim a causal effect of smoking on cancer due to the presence of unmeasured confounding (Genetics). The results are purely associational and may be biased, as they do not account for the influence of genetics on both smoking and cancer.

### Conclusion
While there is an observed association between smoking and cancer, this analysis cannot establish a causal effect due to unmeasured confounding by genetics. Future studies should aim to control for genetic factors to better understand the true causal relationship.

---

## Example 3 — L3 Counterfactual (with SCM)

> "A patient took the drug (X=1) and recovered (Y=0.7). What would their recovery have been had they NOT taken the drug?"

This is a **counterfactual** question → routed to L3. The SCM is fully specified.

In [9]:
result_l3 = await run_causal_workflow(
    question=(
        "A patient took the drug (X=1) and recovered with score Y=0.7. "
        "What would their recovery score have been had they NOT taken the drug (X=0)?"
    ),
    llm=llm,
    dag=drug_dag,
    scm=drug_scm,
)

print("Ladder rung:", result_l3["ladder_rung"])
Markdown(result_l3["final_report"])

Ladder rung: L3


## Causal Report

### Question
A patient took the drug (X=1) and recovered with score Y=0.7. What would their recovery score have been had they NOT taken the drug (X=0)?

### Ladder Rung
This analysis addressed the L3 rung, which focuses on counterfactual estimands. The question seeks to estimate the recovery score had the patient not taken the drug, making it a counterfactual inquiry.

### Causal Query
- **Treatment**: Drug (X)
- **Outcome**: Recovery score (Y)
- **Estimand**: Counterfactual recovery score \( Y_{x=0} \)

### DAG / SCM Summary
The assumed causal structure is represented by a simple Directed Acyclic Graph (DAG) where the drug (X) directly influences the recovery score (Y). There are no confounders affecting this relationship. The Structural Causal Model (SCM) indicates that the recovery score is determined by the drug's effect and an unobserved variable \( U_Y \) that adds variability to the outcome.

### Analysis
- **Method used**: Abduction-Action-Prediction
- **Result / estimate**: The estimated counterfactual recovery score when the drug is not taken (X=0) is 0.6.
- **Confidence / uncertainty**: The confidence interval for this estimate is [0.5, 0.7].

### Assumptions
1. The only effect on Y comes from the drug (X) and the unobserved variable \( U_Y \).
2. The variability in \( U_Y \) is normally distributed with a mean of 0 and a standard deviation of 0.1.
3. There are no unmeasured confounders affecting the relationship between X and Y.

### Validation
The validation checks passed with no issues identified. The identifiability of the causal effect is confirmed, positivity is satisfied, the estimator matches the estimand, and there are no threats from unmeasured confounders.

### Caveats & Limitations
This analysis cannot claim that the estimated recovery score of 0.6 is the definitive outcome for all patients who do not take the drug. The variability in \( U_Y \) suggests that individual outcomes may differ. Additionally, the analysis assumes no other factors influence recovery, which may not hold true in real-world scenarios.

### Conclusion
The estimated recovery score for the patient had they not taken the drug is 0.6, with a confidence interval of [0.5, 0.7]. This suggests that while the drug appears to have a positive effect on recovery, individual outcomes may vary due to unobserved factors.

---

## Example 4 — L3 Counterfactual (without SCM — graceful degradation)

Same counterfactual question, but **no SCM** is provided.
The L3 agent should degrade to bounds (Manski) and communicate the limitation.

In [10]:
result_l3_no_scm = await run_causal_workflow(
    question=(
        "A patient took the drug (X=1) and recovered with score Y=0.7. "
        "What would their recovery score have been had they NOT taken the drug (X=0)?"
    ),
    llm=llm,
    dag=drug_dag,
    scm=None,  # no SCM → agent should degrade to bounds
)

print("Ladder rung:", result_l3_no_scm["ladder_rung"])
Markdown(result_l3_no_scm["final_report"])

Ladder rung: L3


## Causal Report

### Question
A patient took the drug (X=1) and recovered with score Y=0.7. What would their recovery score have been had they NOT taken the drug (X=0)?

### Ladder Rung
L3 was addressed because the question seeks a counterfactual estimate of the recovery score had the patient not taken the drug, which requires a more complex causal analysis involving assumptions about treatment effects.

### Causal Query
- **Treatment**: drug (X)
- **Outcome**: recovery score (Y)
- **Estimand**: counterfactual Y_{x=0}
- **Covariates**: None
- **Assumptions**: None specified

### DAG / SCM Summary
The assumed causal structure is represented by a simple Directed Acyclic Graph (DAG) where the drug (X) directly influences the recovery score (Y). There are no confounders indicated, suggesting a straightforward relationship.

### Analysis
- **Method used**: Bounds
- **Result / estimate**: Unknown
- **Confidence / uncertainty**: Unknown
- **Details**: Without a fully specified Structural Causal Model (SCM), a precise counterfactual estimate for Y_{x=0} cannot be computed. However, bounds can be considered based on monotonicity assumptions. If the treatment (drug) has a non-decreasing effect on recovery score, then Y_{x=0} could be less than or equal to Y=0.7. Conversely, without additional information, a lower bound cannot be determined.

### Assumptions
- The treatment (drug) has a non-decreasing effect on the recovery score.
- The absence of confounders affecting both treatment and outcome.

### Validation
The validation checks revealed several issues:
- **Identifiability**: Not okay; the required full Structural Causal Model (SCM) is missing, which is essential for identifying the counterfactual estimand in L3.
- **Positivity**: Okay; the treatment assignment is valid.
- **Estimator matches estimand**: Not okay; the analysis method (bounds) does not provide a precise estimate for the counterfactual Y_{x=0}.
- **Unmeasured confounders threat**: Not an issue; no unmeasured confounders were identified.
- **Issues**: The lack of a fully specified SCM and the chosen analysis method limits the ability to provide a precise counterfactual estimate.

### Caveats & Limitations
This analysis cannot claim a precise counterfactual estimate for the recovery score without a fully specified SCM. The bounds provided are highly dependent on the assumptions made about the treatment effect, and without additional data or structure, the analysis remains inconclusive.

### Conclusion
While we cannot determine the exact recovery score had the patient not taken the drug, we can infer that it is likely less than or equal to 0.7, assuming the drug has a non-decreasing effect. However, the lack of a complete causal model limits the reliability of this conclusion, and further data is needed for a more definitive estimate.

---

## Inspect Intermediate State

Each run returns the full `CausalState`. You can inspect every intermediate
artifact: the orchestrator's classification, the rung agent's analysis,
the validator's checks, and the final report.

In [11]:
import json

print("=== L2 Example — Full State ===\n")
for key in ("ladder_rung", "causal_query", "analysis_result", "validation_result", "iteration"):
    print(f"── {key} ──")
    val = result_l2.get(key)
    if isinstance(val, dict):
        print(json.dumps(val, indent=2))
    else:
        print(val)
    print()

=== L2 Example — Full State ===

── ladder_rung ──
L1

── causal_query ──
{
  "treatment": "smoking",
  "outcome": "cancer",
  "estimand": "P(cancer | smoking)",
  "covariates": [],
  "assumptions": []
}

── analysis_result ──
{
  "method": "Regression / partial correlation",
  "estimand_type": "associational",
  "estimate": "The analysis would show an association between smoking and cancer, but this association may be confounded by genetics.",
  "confidence_interval": "N/A",
  "details": "In the provided DAG, genetics is a common cause of both smoking and cancer, which indicates that the observed association between smoking and cancer could be spurious due to this confounding factor. To analyze the association, I would use regression to control for the effect of genetics, if it were measured. However, since genetics is unmeasured, the analysis will not fully account for this confounding, leading to a potentially biased estimate of the association.",
  "caveats": [
    "This is associa